# **Documentación del Código: Interacción con APIs de Google Maps**

Este script contiene tres funciones principales diseñadas para interactuar con las APIs de Google Maps. Estas funciones son herramientas clave para enriquecer y validar datos geográficos en aplicaciones que trabajan con direcciones, coordenadas o nombres de lugares.

---

## **1. `obtener_direccion`**
### **Descripción:**
Convierte coordenadas geográficas (latitud y longitud) en una dirección legible. Esto se realiza utilizando la API de Geocoding de Google.

### **¿Cuándo usarla?**
- Cuando tienes coordenadas pero necesitas la dirección asociada.
- Para convertir datos geográficos en información más comprensible.

### **Entrada:**
- `latitud`: Coordenada de latitud.  
- `longitud`: Coordenada de longitud.  
- `api_key`: Clave API de Google.  

### **Salida:**
- Una dirección formateada (texto).  
- `None` si no se encuentra una dirección o hay errores.

---

## **2. `obtener_coordenadas`**
### **Descripción:**
Convierte una dirección en texto en sus coordenadas geográficas (latitud y longitud). Utiliza la API de Geocoding de Google.

### **¿Cuándo usarla?**
- Cuando tienes direcciones pero necesitas coordenadas para análisis espacial o integración con mapas.
- Para validar o enriquecer bases de datos con ubicaciones geográficas.

### **Entrada:**
- `direccion`: Dirección en texto.  
- `api_key`: Clave API de Google.  

### **Salida:**
- Latitud y longitud asociadas a la dirección.  
- `(None, None)` si no se encuentran coordenadas o hay errores.

---

## **3. `obtener_info_local`**
### **Descripción:**
Busca información de un lugar basado en su nombre (como un local comercial, punto de referencia, etc.). Utiliza la API de Places de Google.

### **¿Cuándo usarla?**
- Cuando tienes el nombre de un lugar pero necesitas su dirección o coordenadas.
- Para completar bases de datos donde solo se dispone de nombres de lugares.

### **Entrada:**
- `nombre_local`: Nombre del lugar o punto de referencia.  
- `api_key`: Clave API de Google.  

### **Salida:**
- Dirección completa, latitud y longitud del lugar.  
- `(None, None, None)` si no se encuentra el lugar o hay errores.

---

## **Uso General**
Este código es útil para:
1. **Enriquecimiento de datos**: Completar datos faltantes en bases de datos (direcciones o coordenadas).  
2. **Validación**: Verificar la consistencia entre direcciones y coordenadas.  
3. **Integración con mapas**: Usar datos enriquecidos para integrarlos en visualizaciones espaciales o mapas interactivos.

Las funciones están diseñadas para ser modulares y reutilizables en distintos contextos, facilitando la interacción con las APIs de Google Maps y automatizando tareas comunes relacionadas con datos geográficos.


## Librerias a utilizar y llaves Api
llaves son del servicio de google para utilizar sus api, estas son privadas y estan restringidas y configuradas para su uso, razon por la cual no hay una presente en el codigo (su descuido puede constar recursos)

In [2]:
import requests
import pandas as pd
# Configurar pandas para mostrar todas las columnas
pd.set_option('display.max_columns', 25)
# Tu clave API de Google (delicada)
API_KEY = str(input())

In [3]:
# Función para obtener dirección a partir de latitud y longitud (Geocodificación Inversa)
def obtener_direccion(latitud, longitud, api_key):
    """
    Dado un par de coordenadas (latitud, longitud), obtiene la dirección completa asociada.
    Utiliza la API de Geocoding de Google para realizar la búsqueda.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"  # Endpoint de la API
    parametros = {
        "latlng": f"{latitud},{longitud}",  # Coordenadas a buscar
        "key": api_key  # Clave API
    }
    respuesta = requests.get(url, params=parametros)  # Realiza la solicitud GET
    if respuesta.status_code == 200:  # Verifica si la solicitud fue exitosa
        datos = respuesta.json()  # Parseo de la respuesta a JSON
        if datos['status'] == "OK":  # Verifica si la API devolvió resultados válidos
            return datos['results'][0]['formatted_address']  # Retorna la dirección formateada
        else:
            return None  # Error en la API (sin resultados válidos)
    else:
        return None  # Error en la conexión o solicitud fallida

# Función para obtener latitud y longitud a partir de una dirección (Geocodificación)
def obtener_coordenadas(direccion, api_key):
    """
    Dada una dirección, obtiene las coordenadas geográficas (latitud y longitud).
    Utiliza la API de Geocoding de Google para realizar la búsqueda.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"  # Endpoint de la API
    parametros = {
        "address": direccion,  # Dirección a buscar
        "key": api_key  # Clave API
    }
    respuesta = requests.get(url, params=parametros)  # Realiza la solicitud GET
    if respuesta.status_code == 200:  # Verifica si la solicitud fue exitosa
        datos = respuesta.json()  # Parseo de la respuesta a JSON
        if datos['status'] == "OK":  # Verifica si la API devolvió resultados válidos
            coordenadas = datos['results'][0]['geometry']['location']  # Obtiene las coordenadas
            return coordenadas['lat'], coordenadas['lng']  # Retorna latitud y longitud
        else:
            return None, None  # Error en la API (sin resultados válidos)
    else:
        return None, None  # Error en la conexión o solicitud fallida

# Función para obtener información completa de un local dado su nombre
def obtener_info_local(nombre_local, api_key):
    """
    Dado el nombre de un local, obtiene su dirección completa y coordenadas geográficas.
    Utiliza la API de Places de Google para realizar la búsqueda.
    """
    url = "https://maps.googleapis.com/maps/api/place/findplacefromtext/json"  # Endpoint de la API Places
    
    # Parámetros de la solicitud para buscar el local
    parametros = {
        "input": nombre_local,           # Nombre del lugar (texto a buscar)
        "inputtype": "textquery",        # Tipo de búsqueda
        "fields": "formatted_address,geometry",  # Campos requeridos: dirección y geometría
        "key": api_key  # Clave API
    }
    
    respuesta = requests.get(url, params=parametros)  # Realiza la solicitud GET
    
    if respuesta.status_code == 200:  # Verifica si la solicitud fue exitosa
        datos = respuesta.json()  # Parseo de la respuesta a JSON
        if datos['status'] == "OK":  # Verifica si la API devolvió resultados válidos
            resultado = datos['candidates'][0]  # Toma el primer resultado de la búsqueda
            direccion = resultado.get("formatted_address", None)  # Obtiene la dirección formateada
            coordenadas = resultado.get("geometry", {}).get("location", None)  # Obtiene las coordenadas
            latitud = coordenadas.get("lat") if coordenadas else None  # Latitud del local
            longitud = coordenadas.get("lng") if coordenadas else None  # Longitud del local
            return direccion, latitud, longitud  # Retorna dirección, latitud y longitud
        else:
            print(f"Error en la API: {datos['status']}")  # Mensaje de error de la API
            return None, None, None  # Sin resultados válidos
    else:
        print(f"Error de conexión: {respuesta.status_code}")  # Mensaje de error de conexión
        return None, None, None  # Solicitud fallida


# **Documentación: Funcionalidades de la API de Google Maps**

La API de Google Maps ofrece una amplia variedad de funcionalidades relacionadas con direcciones, coordenadas y lugares. Además de convertir direcciones en coordenadas, realizar geocodificación inversa y buscar información a partir del nombre de un lugar, permite obtener otros datos útiles. A continuación, se describen las principales capacidades y limitaciones.

---

## **1. Funcionalidades principales relacionadas con direcciones**

### **1.1. Geocodificación (Dirección → Coordenadas)**
- Convierte una dirección en texto en coordenadas geográficas (latitud y longitud).
- **Uso típico:** Enriquecer bases de datos o integrar datos en mapas.

### **1.2. Geocodificación inversa (Coordenadas → Dirección)**
- Convierte coordenadas geográficas en una dirección formateada.
- **Uso típico:** Mostrar información legible al usuario basada en su ubicación.

### **1.3. Búsqueda de lugares por nombre**
- Permite obtener la dirección completa, latitud y longitud de un lugar o punto de interés basado únicamente en su nombre.
- **Uso típico:** Completar datos o integrar con aplicaciones de búsqueda.

---

**Ejemplos:**

## Sin Direccion (antes)

In [4]:
dfsd = pd.read_excel("Ejemplo data Periferia.xlsx", sheet_name='Sin Direcciones')
dfsd.head(2)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,SIN DIRECCION,CADENAS,ESPECIALIZADOS,NO APLICA,5344202,-72381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781
1,Único,153545,Moderno,GROUPE SEB,1007907,EXITO ORIENTAL BUCARAMANGA CV,SIN DIRECCION,CADENAS,SUPER,NO APLICA,7099395,-73106722,1,NaN,42,1007907,2,BUCARAMANGA,SANTANDER,569800,EXITO ORIENTAL BUCARAMANGA CV-BUCARAMANGA-SANT...,42,569800


In [ ]:
#limpieza
dfsd['LATITUD'] = dfsd['LATITUD'].apply(lambda x: x / 1000000 if x != 200 else x)
dfsd['LONGITUD'] = dfsd['LONGITUD'].apply(lambda x: x / 1000000 if x != 200 else x)

In [ ]:
dfsd['DIRECCION'] = dfsd.apply(lambda row: obtener_direccion(row['LATITUD'], row['LONGITUD'], API_KEY), axis=1)  #deberiamos aplicarle un if Sin direccion? o un if not null? creo que una exploracion de datos seria mas eficiente'


## Sin Direccion (Despues)

In [6]:
dfsd.head(2)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,"Cl. 24 con carrera 35, Yopal, Casanare, Colombia",CADENAS,ESPECIALIZADOS,NO APLICA,5344202,-72381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781
1,Único,153545,Moderno,GROUPE SEB,1007907,EXITO ORIENTAL BUCARAMANGA CV,"Viad. La Flora, Sotomayor, Bucaramanga, Santan...",CADENAS,SUPER,NO APLICA,7099395,-73106722,1,NaN,42,1007907,2,BUCARAMANGA,SANTANDER,569800,EXITO ORIENTAL BUCARAMANGA CV-BUCARAMANGA-SANT...,42,569800


## Sin Latitud Ni Longitud (Antes)

In [7]:
dfslt = pd.read_excel("Ejemplo data Periferia.xlsx", sheet_name='Sin Lat-long')
dfslt.head(6)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,120889,Moderno,KCC PHARMA,313304,BOTICA COLINAS,CR 38 100 09 LC 1,KCC PHARMA,MODERNO,CADENA,NaN,NaN,1,NaN,1,313304,1,BARRANQUILLA,ATLÁNTICO,313304,BOTICA COLINAS-BARRANQUILLA-ATLÁNTICO,1,313304
1,Único,148307,Moderno,MERCK SA,389602,DROGUERIA FARMASANITAS CC GUATAPURI VALLEDUPAR,CRA 9 13 C 80,MERCK,MODERNO,NO APLICA,NaN,NaN,1,NaN,1,389602,1,VALLEDUPAR,CESAR,389602,DROGUERIA FARMASANITAS CC GUATAPURI VALLEDUPAR...,1,389602
2,Único,49412,Moderno,BSN MEDICAL,394203,COLSUBSIDIO DEP CABRERA,CL 90 11A 40 LC 6,RETAIL/MODERNO,8807,PRESENCIAL,200.0,200.0,1,NaN,1,394203,1,BOGOTÁ,BOGOTA D.C,394203,COLSUBSIDIO DEP CABRERA-BOGOTÁ-BOGOTA D.C,1,394203
3,Único,119635,Moderno,NACIONAL CHOCOLATES - CI,403611,CIPEREIRA,AV 30 DE AGOSTO 36 - 10,NACIONAL CHOCOLATES - CI,MODERNO,VACIO,NaN,NaN,1,NaN,2,403611,2,PEREIRA,RISARALDA,403139,CIPEREIRA-PEREIRA-RISARALDA,2,403139
4,Único,34127,Moderno,AZUL K,432825,AUTOSERVICIO LA,CRA 4 # 5 - 39,AZULK,CANAL TRADICIONAL,131,200.0,200.0,1,LA 14,20,432825,1,SUAZA,HUILA,649999,AUTOSERVICIO LA-SUAZA-HUILA,1,432825


# Posibles valores para pedir latitud y longitud

In [8]:
dfslt['LATITUD'], dfslt['LONGITUD'] = zip(*dfslt.apply(lambda row: obtener_coordenadas(row['DIRECCION'] + ' , ' + row['CIUDAD'] + ' , ' + row['DEPARTAMENTO'] + ' , COLOMBIA', API_KEY), axis=1)) #FUNCIONA!

In [9]:
#dfslt['LATITUD'], dfslt['LONGITUD'] = zip(*dfslt.apply(lambda row: obtener_coordenadas(row['DIRECCION'] + ' , ' + row['CIUDAD'] + ' , ' + row['DEPARTAMENTO'], API_KEY), axis=1))

In [10]:
#dfslt['LATITUD'], dfslt['LONGITUD'] = zip(*dfslt.apply(lambda row: obtener_coordenadas(row['DIRECCION'] + ' , ' + row['CIUDAD'] + ' , COLOMBIA', API_KEY), axis=1))

In [11]:
#dfslt['LATITUD'], dfslt['LONGITUD'] = zip(*dfslt.apply(lambda row: obtener_coordenadas(row['DIRECCION'] + ' , ' + row['CIUDAD'], API_KEY), axis=1))

## Sin Latitud Ni Longitud (Antes)

In [12]:
dfslt.head(6)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,120889,Moderno,KCC PHARMA,313304,BOTICA COLINAS,CR 38 100 09 LC 1,KCC PHARMA,MODERNO,CADENA,10.982847,-74.832692,1,NaN,1,313304,1,BARRANQUILLA,ATLÁNTICO,313304,BOTICA COLINAS-BARRANQUILLA-ATLÁNTICO,1,313304
1,Único,148307,Moderno,MERCK SA,389602,DROGUERIA FARMASANITAS CC GUATAPURI VALLEDUPAR,CRA 9 13 C 80,MERCK,MODERNO,NO APLICA,10.477112,-73.248803,1,NaN,1,389602,1,VALLEDUPAR,CESAR,389602,DROGUERIA FARMASANITAS CC GUATAPURI VALLEDUPAR...,1,389602
2,Único,49412,Moderno,BSN MEDICAL,394203,COLSUBSIDIO DEP CABRERA,CL 90 11A 40 LC 6,RETAIL/MODERNO,8807,PRESENCIAL,4.672903,-74.049876,1,NaN,1,394203,1,BOGOTÁ,BOGOTA D.C,394203,COLSUBSIDIO DEP CABRERA-BOGOTÁ-BOGOTA D.C,1,394203
3,Único,119635,Moderno,NACIONAL CHOCOLATES - CI,403611,CIPEREIRA,AV 30 DE AGOSTO 36 - 10,NACIONAL CHOCOLATES - CI,MODERNO,VACIO,4.812387,-75.708940,1,NaN,2,403611,2,PEREIRA,RISARALDA,403139,CIPEREIRA-PEREIRA-RISARALDA,2,403139
4,Único,34127,Moderno,AZUL K,432825,AUTOSERVICIO LA,CRA 4 # 5 - 39,AZULK,CANAL TRADICIONAL,131,1.976566,-75.794315,1,LA 14,20,432825,1,SUAZA,HUILA,649999,AUTOSERVICIO LA-SUAZA-HUILA,1,432825


## Sin direccion ni latitut ni longitud (Antes)

In [13]:
dfsdlt = pd.read_excel("Ejemplo data Periferia.xlsx", sheet_name='SinDirecc, Lat-Long')
dfsdlt.head(2)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,62442,Moderno,HARINERA DEL VALLE S A,298763,GARCIA VALENZUELA ALFONSO,SIN DIRECCIÓN,INDEPENDIENTES,CADENAS REGIONALES/SMI,VACIO,200.0,200.0,1,NaN,1,298763,1,FUNZA,CUNDINAMARCA,298763,GARCIA VALENZUELA ALFONSO-FUNZA-CUNDINAMARCA,1,298763
1,Único,67593,Moderno,RB COLOMBIA SA,847696,SUPER INTEREX INDEPENDENCIA,SIN DIRECCION,RECKITT BENCKISER COLOMBIA SA,CANAL MODERNO,NaN,200.0,200.0,1,NaN,1,847696,1,SANTIAGO DE CALI,VALLE DEL CAUCA,847696,SUPER INTEREX INDEPENDENCIA-SANTIAGO DE CALI-V...,1,847696


In [14]:
#dfsdlt['DIRECCION'], dfsdlt['LATITUD'], dfsdlt['LONGITUD'] = zip(*dfsdlt['NOMBRE_CLIENTE'].apply(lambda x: obtener_info_local(x, API_KEY))) #podriamos colocar el nombre del cliente si el del provedor no aparece

In [15]:
dfsdlt['DIRECCION'], dfsdlt['LATITUD'], dfsdlt['LONGITUD'] = zip(*dfsdlt.apply(lambda row: obtener_info_local(f"{row['NOMBRE_CLIENTE']}, {row['CIUDAD']}, {row['DEPARTAMENTO']}", API_KEY),axis=1))

## Sin direccion ni latitut ni longitud (Despues)

In [16]:
dfsdlt.head(6)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,62442,Moderno,HARINERA DEL VALLE S A,298763,GARCIA VALENZUELA ALFONSO,"Cra. 34 #13-45, Bogotá, Colombia",INDEPENDIENTES,CADENAS REGIONALES/SMI,VACIO,4.617797,-74.094966,1,NaN,1,298763,1,FUNZA,CUNDINAMARCA,298763,GARCIA VALENZUELA ALFONSO-FUNZA-CUNDINAMARCA,1,298763
1,Único,67593,Moderno,RB COLOMBIA SA,847696,SUPER INTEREX INDEPENDENCIA,"Cl. 46 #5-76, COMUNA 4, Cali, Valle del Cauca,...",RECKITT BENCKISER COLOMBIA SA,CANAL MODERNO,NaN,3.460963,-76.504099,1,NaN,1,847696,1,SANTIAGO DE CALI,VALLE DEL CAUCA,847696,SUPER INTEREX INDEPENDENCIA-SANTIAGO DE CALI-V...,1,847696
2,Único,150359,Moderno,SC JOHNSON,440780,EXITO ESTADIO NORTE,"Cl. 93 #13 45, Bogotá, Colombia",SC JOHNSON,CANAL MODERNO,NO APLICA,4.675649,-74.050290,1,NaN,7,440780,1,MEDELLIN,ANTIOQUIA,1007793,EXITO ESTADIO NORTE-MEDELLIN-ANTIOQUIA,7,1007793
3,Único,6799,Moderno,inactivo,1229768,CALI VARGAS IPIALES,"Cra. 5 #16-39, Centro, Ipiales, Nariño, Colombia",MODERNO,ESPECIALIZADO,NO APLICA,0.827336,-77.640889,1,NaN,1,1229768,1,IPIALES,NARIÑO,1229768,CALI VARGAS IPIALES-IPIALES-NARIÑO,1,1229768
4,Único,79402,Moderno,BSN MEDICAL,1410761,CAFAM CAJICA,"Edificio Patmer, Cl. 3 #Nº 10 – 66, Cajicá, Cu...",MODERNO,SB,PRESENCIAL,4.919216,-74.030540,1,NaN,1,1410761,1,CAJICÁ,CUNDINAMARCA,1410761,CAFAM CAJICA-CAJICÁ-CUNDINAMARCA,1,1410761


In [33]:
dfrv = pd.read_excel("Ejemplo data Periferia.xlsx", sheet_name='Revisar')
dfrv.head(6)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,126762,Moderno,CENTRAL CERVECERA DE COLOMBIA,373462,TP LA ESTACION,KR 1 37 36,MODERNO,HEINEKEN,POSTOBON YUMBO,3465616,-76513349,1,NaN,1,373462,1,SANTIAGO DE CALI,VALLE DEL CAUCA,373462,TP LA ESTACION-SANTIAGO DE CALI-VALLE DEL CAUCA,1,373462
1,Único,49412,Moderno,BSN MEDICAL,394203,COLSUBSIDIO DEP CABRERA,CL 90 11A 40 LC 6,RETAIL/MODERNO,8807,PRESENCIAL,200,200,1,NaN,1,394203,1,BOGOTÁ,BOGOTA D.C,394203,COLSUBSIDIO DEP CABRERA-BOGOTÁ-BOGOTA D.C,1,394203
2,Único,34127,Moderno,AZUL K,432825,AUTOSERVICIO LA,CRA 4 # 5 - 39,AZULK,CANAL TRADICIONAL,131,200,200,1,LA 14,20,432825,1,SUAZA,HUILA,649999,AUTOSERVICIO LA-SUAZA-HUILA,1,432825
3,Único,19333,Moderno,NUTRESA IMPULSO - MERCADEO,465918,GALERIA ALAMEDA STO,CL 9 23C 58,CADENAS,GRANDES CADENAS,NO APLICA,200,200,1,NaN,1,465918,1,SANTIAGO DE CALI,VALLE DEL CAUCA,465918,GALERIA ALAMEDA STO-SANTIAGO DE CALI-VALLE DEL...,1,465918
4,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,Calle 90 11A 40 Local 6,CADENAS,ESPECIALIZADOS,NO APLICA,5344202,-72381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781
5,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,Carrea 4 # 5 - 39,CADENAS,ESPECIALIZADOS,NO APLICA,5344202,-72381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781


In [34]:
#limpieza
dfrv['LATITUD'] = dfrv['LATITUD'].apply(lambda x: x / 1000000 if x != 200 else x)
dfrv['LONGITUD'] = dfrv['LONGITUD'].apply(lambda x: x / 1000000 if x != 200 else x)

In [35]:
dfrv['LATITUD'], dfrv['LONGITUD'] = zip(*dfrv.apply(lambda row: obtener_coordenadas(
    row['DIRECCION'] + ' , ' + row['CIUDAD'] + ' , ' + row['DEPARTAMENTO'] + ' , COLOMBIA', API_KEY)
    if (row['LATITUD'] == 200 and row['LONGITUD'] == 200) else (row['LATITUD'], row['LONGITUD']), axis=1))

In [37]:
dfrv['DIRECCION'] = dfrv.apply(lambda row: obtener_direccion(row['LATITUD'], row['LONGITUD'], API_KEY), axis=1) 

In [38]:
dfrv.head(6)

,UNIFICADO,F1,CANAL,NOMBRE_CLIENTE,PUNTO_VENTA_ID,NOMBRE_PDV,DIRECCION,NOMBRE_SEGMENTO_1,NOMBRE_SEGMENTO_2,NOMBRE_SEGMENTO_3,LATITUD,LONGITUD,EST_ACTIVO,OBSERVACIONES,CANTIDAD,PUNTO_VENTA_ID - TEXTO,DUPLICADOS,CIUDAD,DEPARTAMENTO,ID_UNICO,PDV_TOTAL,CANTIDAD_TOTAL,ID_UNICO_TOTAL
0,Único,126762,Moderno,CENTRAL CERVECERA DE COLOMBIA,373462,TP LA ESTACION,"Cra. 1 #3804, Esmeralda, Cali, Valle del Cauca...",MODERNO,HEINEKEN,POSTOBON YUMBO,3.465616,-76.513349,1,NaN,1,373462,1,SANTIAGO DE CALI,VALLE DEL CAUCA,373462,TP LA ESTACION-SANTIAGO DE CALI-VALLE DEL CAUCA,1,373462
1,Único,49412,Moderno,BSN MEDICAL,394203,COLSUBSIDIO DEP CABRERA,"Cl. 90 #11a-42, Bogotá, Colombia",RETAIL/MODERNO,8807,PRESENCIAL,4.672903,-74.049876,1,NaN,1,394203,1,BOGOTÁ,BOGOTA D.C,394203,COLSUBSIDIO DEP CABRERA-BOGOTÁ-BOGOTA D.C,1,394203
2,Único,34127,Moderno,AZUL K,432825,AUTOSERVICIO LA,"Cra. 4 # 5-55, Suaza, Huila, Colombia",AZULK,CANAL TRADICIONAL,131,1.976566,-75.794315,1,LA 14,20,432825,1,SUAZA,HUILA,649999,AUTOSERVICIO LA-SUAZA-HUILA,1,432825
3,Único,19333,Moderno,NUTRESA IMPULSO - MERCADEO,465918,GALERIA ALAMEDA STO,"Cra 52A #95, Las Canas, Cali, Valle del Cauca,...",CADENAS,GRANDES CADENAS,NO APLICA,3.414904,-76.559519,1,NaN,1,465918,1,SANTIAGO DE CALI,VALLE DEL CAUCA,465918,GALERIA ALAMEDA STO-SANTIAGO DE CALI-VALLE DEL...,1,465918
4,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,"Cl. 24 con carrera 35, Yopal, Casanare, Colombia",CADENAS,ESPECIALIZADOS,NO APLICA,5.344202,-72.381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781
5,Único,153433,Moderno,GROUPE SEB,1008511,HOMECENTER YOPAL,"Cl. 24 con carrera 35, Yopal, Casanare, Colombia",CADENAS,ESPECIALIZADOS,NO APLICA,5.344202,-72.381242,1,NaN,15,1008511,3,YOPAL,CASANARE,509781,HOMECENTER YOPAL-YOPAL-CASANARE,13,509781


In [19]:
with pd.ExcelWriter('DataCorregida.xlsx', engine='openpyxl') as writer:
    dfsdlt.to_excel(writer, sheet_name='SinDirecc, Lat-Long', index=False)
    dfslt.to_excel(writer, sheet_name='Sin Lat-long', index=False)
    dfsd.to_excel(writer, sheet_name='Sin Direcciones', index=False)
    #df4.to_excel(writer, sheet_name='Hoja4', index=False)